# TrustGuard: prediction notebook (Track 2)

Runs on the platform with **no internet** and **no train.csv**: it only loads `model.pkl`, builds the same features as in training, and predicts.

* `model.pkl` holds `(model, features)`: a scikit-learn pipeline trained in our training notebook `TrustGuard_Track2.ipynb` with the platform's library versions:
  * median imputation of missing numbers,
  * continuous inputs capped at the 1st and 99th training percentiles, and the amount features capped at their 90th percentile, so the extreme values of the shifted test set cannot dominate a score,
  * 0/1 flags (including the missing-value flags) left uncapped,
  * scaling, fixed one-hot encoding of the categories, and a logistic regression with `C = 3`.
* `make_features` below is the exact, stateless feature engineering used in training. It works on rows in any order, without an `id` column, and with missing or unexpected values.
* Output: one `prediction` per row (probability of fraud, 0 to 1), in the same order as the input file.


In [ ]:
import os
import joblib
import pandas as pd

TEST_PATH = os.environ.get("DATATHON_INPUT_PATH", "test.csv")
OUTPUT_PATH = os.environ.get("DATATHON_OUTPUT_PATH", "predictions.csv")

In [ ]:
"""The SAME feature engineering as in our training notebook (stateless: nothing is learned from the data here)."""
import numpy as np
import pandas as pd

RAW_NUM = ['transaction_amount', 'transaction_hour', 'transactions_last_24h', 'spend_last_24h',
           'account_age', 'new_device', 'transactions_last_1h']
CAT_COLS = ['merchant_category', 'country', 'transaction_channel']
CAT_LEVELS = {
    'merchant_category': ['grocery', 'transport', 'retail', 'entertainment', 'utilities', 'gaming',
                          'food_delivery', 'travel', 'electronics', 'cash_transfer', 'luxury'],
    'country': ['SG', 'MY', 'TH', 'ID', 'VN', 'PH', 'JP', 'US', 'GB', 'AU'],
    'transaction_channel': ['card_present', 'mobile_app', 'ecommerce', 'bank_transfer'],
}
TX1H_LABELS = ['0-2', '3', '4', '5', '6', '7+']
# 0.1% and 99.9% percentiles of the training data (fixed constants, so the function stays stateless)
CLIP = {'transaction_amount': (2.5, 5593.7418), 'spend_last_24h': (0.0, 8906.8026), 'account_age': (5.0, 4000.0),
        'transactions_last_24h': (1.0, 26.0), 'transactions_last_1h': (0.0, 9.0)}
NA_COLS = ['transactions_last_24h', 'spend_last_24h', 'account_age', 'new_device', 'transactions_last_1h']


def make_features(df):
    if isinstance(df, dict):            # a single transaction given as {column: value}
        df = pd.DataFrame([df]) if all(np.isscalar(v) or v is None for v in df.values()) else pd.DataFrame(df)
    elif isinstance(df, pd.Series):
        df = df.to_frame().T
    df = pd.DataFrame(df).copy()
    df.columns = [str(c).strip().lower().replace(' ', '_') for c in df.columns]
    df = df.reset_index(drop=True)
    d = pd.DataFrame(index=df.index)
    # cleaning: numbers
    for c in RAW_NUM:
        d[c] = pd.to_numeric(df[c], errors='coerce') if c in df else np.nan
    d = d.astype(float).replace([np.inf, -np.inf], np.nan)
    for c in ['transaction_amount', 'transactions_last_24h', 'spend_last_24h', 'account_age', 'transactions_last_1h']:
        d[c] = d[c].clip(lower=0)
    d['transaction_hour'] = np.floor(d['transaction_hour'] % 24)
    d['new_device'] = d['new_device'].clip(0, 1).round()
    for c, (lo, hi) in CLIP.items():
        d[c] = d[c].clip(lo, hi)
    d['transactions_last_24h'] = np.fmax(d['transactions_last_24h'], d['transactions_last_1h'])
    # cleaning: categories (unseen or missing -> 'unknown', which every encoder treats as all-zero)
    for c in CAT_COLS:
        s = df[c] if c in df else pd.Series(np.nan, index=df.index)
        s = s.astype('string').str.strip()
        s = s.str.upper() if c == 'country' else s.str.lower().str.replace(' ', '_')
        allowed = set(CAT_LEVELS[c])
        d[c] = [v if v in allowed else 'unknown' for v in s.to_numpy(dtype=object, na_value=None)]
    known = lambda s: s.notna()
    # velocity
    d['vel_mult'] = np.log1p(d.transactions_last_1h / (d.transactions_last_24h / 24 + 0.1))
    d['tx1h_share'] = d.transactions_last_1h / (d.transactions_last_24h + 1)
    d['burst'] = (d.transactions_last_1h >= 4).astype(float).where(known(d.transactions_last_1h))
    band = pd.cut(d.transactions_last_1h, [-np.inf, 2, 3, 4, 5, 6, np.inf], labels=TX1H_LABELS).astype(object)
    d['tx1h_band'] = band.where(band.notna(), '0-2').astype(str)
    # monetary
    d['log_amount'] = np.log1p(d.transaction_amount)
    d['log_spend24'] = np.log1p(d.spend_last_24h)
    d['log_tx24'] = np.log1p(d.transactions_last_24h)
    avg24 = d.spend_last_24h / (d.transactions_last_24h + 1)
    d['amt_vs_avg24'] = d.log_amount - np.log1p(avg24)
    d['amt_share'] = d.transaction_amount / (d.spend_last_24h + d.transaction_amount + 1)
    d['amt_vs_age'] = d.log_amount - np.log1p(d.account_age)
    d['spend_intensity'] = d.log_spend24 - np.log1p(d.account_age)
    # identity and trust
    d['log_age'] = np.log1p(d.account_age)
    d['young'] = (d.account_age < 90).astype(float).where(known(d.account_age))
    d['nd_young'] = d.new_device * d.young
    # context
    d['hour_sin'] = np.sin(2 * np.pi * d.transaction_hour / 24)
    d['hour_cos'] = np.cos(2 * np.pi * d.transaction_hour / 24)
    d['night'] = (d.transaction_hour <= 5).astype(float).where(known(d.transaction_hour))
    d['remote'] = d.transaction_channel.isin(['ecommerce', 'mobile_app', 'bank_transfer']).astype(float)
    d['liquid'] = d.merchant_category.isin(['luxury', 'electronics', 'cash_transfer']).astype(float)
    d['foreign'] = ((d.country != 'SG') & (d.country != 'unknown')).astype(float)
    # data quality
    for c in NA_COLS:
        d[c + '_na'] = d[c].isna().astype(float)
    return d


In [ ]:
model, features = joblib.load("model.pkl")  # uploaded in card 2, next to this notebook
X_test = make_features(pd.read_csv(TEST_PATH))[features]
if hasattr(model, "predict_proba"):  # Track 2: probability of fraud
    preds = model.predict_proba(X_test)[:, 1]
else:  # Track 1
    preds = model.predict(X_test)

In [ ]:
pd.DataFrame({"prediction": preds}).to_csv(OUTPUT_PATH, index=False)
print(f"wrote {len(preds)} predictions to {OUTPUT_PATH}")